In [1]:
"""
Configuration de l'environnement — Analyse e-commerce
===================================================
Imports et configuration pour l'analyse exploratoire des données de churn.
"""

import os
import sys

# Manipulation des données
import pandas as pd
import numpy as np

# Statistiques et tests
import scipy.stats as ss

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Seed pour la reproductibilité
RANDOM_STATE = 1204

# Import de la fonction utilitaire depuis utils/data_prep.py
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils.data_prep import detect_possible_outliers

print(f" Imports chargés avec succès")

 Imports chargés avec succès


In [2]:
# Vérification du répertoire de travail
print(f" Répertoire courant : {os.getcwd()}")

 Répertoire courant : c:\Users\juber\Documents\Analyse-e-commerce-GCP-BigQuery\powerbi


In [3]:
# ============================================================
# Chargement du dataset brut
# ============================================================
df = pd.read_csv("..\\data\\thelook_fr_women_2023_2024.csv")
print(f" Dataset importé : {df.shape[0]} lignes et {df.shape[1]} colonnes\n")

 Dataset importé : 1679 lignes et 20 colonnes



In [4]:
# Conversion des dates en datetime
df["order_created_at"] = pd.to_datetime(df["order_created_at"])
df["year"] = df["order_created_at"].dt.year
df["month"] = df["order_created_at"].dt.month
df["month_str"] = df["order_created_at"].dt.strftime("%b")  # nom du mois (Jan, Feb...)

# Création de KPI intermédiaires
df["margin"] = df["sale_price"] - df["cost"]  # Marge brute par ligne
df["is_sale"] = df["order_status"] == "Complete"
df["is_return"] = df["order_status"] == "Returned"

In [5]:
def compute_kpis(df_filtered):
    """Retourne les principaux KPI pour un DataFrame filtré."""
    sales = df_filtered[df_filtered["is_sale"]]
    
    # Chiffre d’affaires
    total_ca = sales["sale_price"].sum()
    
    # Marge brute
    total_margin = sales["margin"].sum()
    
    # Panier moyen
    num_orders = sales["order_id"].nunique()
    avg_basket = total_ca / num_orders if num_orders else 0
    
    # Taux de retour
    returns = df_filtered[df_filtered["is_return"]]
    total_sold_plus_returns = df_filtered[df_filtered["is_sale"] | df_filtered["is_return"]]
    return_rate = len(returns) / len(total_sold_plus_returns) * 100 if len(total_sold_plus_returns) else 0
    
    # Taux de ré-achat
    orders_per_client = sales.groupby(["user_id", "year"])["order_id"].nunique().reset_index(name="nb_orders")
    repeat_customers = orders_per_client[orders_per_client["nb_orders"] >= 2]
    total_clients = sales["user_id"].nunique()
    repeat_rate = len(repeat_customers["user_id"].unique()) / total_clients * 100 if total_clients else 0
    
    return {
        "Chiffre d’affaires (€)": total_ca,
        "Marge brute (€)": total_margin,
        "Panier moyen (€)": avg_basket,
        "Taux de retour (%)": return_rate,
        "Taux de ré-achat (%)": repeat_rate
    }

In [6]:
# Agrégation mensuelle
monthly = df[df["is_sale"]].groupby(["year", "month"])["sale_price"].sum().reset_index()

# Pivot pour index 2023 = 100
pivot = monthly.pivot(index="month", columns="year", values="sale_price").fillna(0)
pivot["index_2023"] = pivot[2023] / pivot[2023].max() * 100
pivot["index_2024"] = pivot[2024] / pivot[2023].max() * 100

pivot_reset = pivot.reset_index()

# Graphique
fig_monthly = px.line(
    pivot_reset,
    x="month",
    y=["index_2023", "index_2024"],
    markers=True,
    labels={"value": "CA indexé (2023=100)", "month": "Mois"},
    title="Comparaison mensuelle 2023 vs 2024 (index 2023=100)"
)
fig_monthly.show()

In [7]:
# ------------------------------
# Top 10 marques par chiffre d'affaires
# ------------------------------

# On filtre uniquement les ventes et on calcule le CA par marque
top_brands = (
    df[df["is_sale"]]                       # On ne garde que les ventes
    .groupby("brand")["sale_price"]         # On regroupe par marque
    .sum()                                  # Somme des ventes
    .sort_values(ascending=True)            # Tri croissant pour barh
    .tail(10)                               # On prend le top 10
)

# Création du graphique horizontal
fig_brand = px.bar(
    top_brands,
    x=top_brands.values,                    # Valeurs du CA
    y=top_brands.index,                     # Marques
    orientation="h",                        # Bar chart horizontal
    text=top_brands.values,                 # Affichage des valeurs sur les barres
    title="Top 10 marques par CA (€)"       # Titre
)

# Ajout des labels axes
fig_brand.update_layout(
    yaxis_title="Marques",
    xaxis_title="CA (€)"
)

# Affichage
fig_brand.show()


# ------------------------------
# Top 10 catégories par marge
# ------------------------------

# On filtre uniquement les ventes et on calcule la marge par catégorie
top_cats = (
    df[df["is_sale"]]
    .groupby("category")["margin"]          # Somme des marges par catégorie
    .sum()
    .sort_values(ascending=True)            # Tri croissant pour barh
    .tail(10)                               # On prend le top 10
)

# Création du graphique horizontal
fig_cat = px.bar(
    top_cats,
    x=top_cats.values,                      # Valeurs de la marge
    y=top_cats.index,                       # Catégories
    orientation="h",                        # Bar chart horizontal
    text=top_cats.values,                   # Affichage des valeurs sur les barres
    title="Top 10 catégories par marge (€)" # Titre
)

# Ajout des labels axes
fig_cat.update_layout(
    yaxis_title="Catégories",
    xaxis_title="Marge (€)"
)

# Affichage
fig_cat.show()

In [8]:
returns_rate_by_brand = df.groupby("brand").apply(
    lambda x: pd.Series({
        "ca": x[x["is_sale"]]["sale_price"].sum(),
        "return_rate": (
            len(x[x["is_return"]]) / len(x[x["is_sale"] | x["is_return"]]) * 100
            if len(x[x["is_sale"] | x["is_return"]]) > 0 else 0
        )
    })
).reset_index()

In [11]:
# ------------------------------
# Création du scatter plot amélioré
# ------------------------------
fig_scatter = px.scatter(
    returns_rate_by_brand,
    x="ca",                                         # Axe X : chiffre d'affaires
    y="return_rate",                                # Axe Y : taux de retour
    size="ca",                                      # Taille des points proportionnelle au CA
    size_max=40,                                   # Taille max pour lisibilité
    color="return_rate",                            # Couleur selon le taux de retour
    color_continuous_scale="RdYlGn_r",            # Palette rouge ↔ vert
    hover_name="brand",                             # Affichage marque au survol
    hover_data={"ca": ":,.0f", "return_rate": ":.2f"},  # Format lisible
    title="Relation CA ↔ taux de retour par marque"
)

# Ajout des labels axes
fig_scatter.update_layout(
    xaxis_title="Chiffre d'affaires (€)",
    yaxis_title="Taux de retour (%)",
    coloraxis_colorbar=dict(title="Taux de retour (%)")
)

# Optionnel : annotation des 5 marques avec CA le plus élevé
top5 = returns_rate_by_brand.nlargest(5, "ca")
for i, row in top5.iterrows():
    fig_scatter.add_annotation(
        x=row["ca"],
        y=row["return_rate"],
        text=row["brand"],
        showarrow=True,
        arrowhead=2,
        ax=20,
        ay=-20
    )

# Affichage
fig_scatter.show()

In [10]:
# Nombre d’articles par commande
basket = df[df["is_sale"]].groupby("order_id")["order_item_id"].count().reset_index(name="nb_items")

fig_basket = px.histogram(
    basket,
    x="nb_items",
    nbins=10,
    title="Distribution du nombre d’articles par commande (panier moyen)"
)
fig_basket.show()